# LAIGO TextGen System Prompt Testing Notebook

This notebook allows you to test and refine the 6 different system prompts for the LAIGO legal AI assistant before hardcoding them into the `text_generation` Lambda function.

## Subroutes Covered:
1. **Intake & Facts** - Gathering case information from the client
2. **Issue Identification** - Identifying legal issues in the case
3. **Research Strategy** - Planning legal research approach
4. **Argument Construction** - Building legal arguments
5. **Contrarian Analysis** - Analyzing opposing arguments and weaknesses
6. **Policy Context** - Understanding policy implications

## Usage:
1. Run the setup cells to initialize Bedrock
2. Edit the system prompts in Section 3
3. Use the test cells for each subroute to refine prompts
4. Copy finalized prompts back to the Lambda function

## 1. Import Required Libraries and Setup

In [ ]:
# Install required packages (run once)
# !pip install boto3 langchain langchain-aws langchain-community ipywidgets

import boto3
import json
from typing import Dict, List, Optional
from dataclasses import dataclass, field
from datetime import datetime

# LangChain imports
from langchain_aws import ChatBedrock
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.chat_history import InMemoryChatMessageHistory

print("✅ Libraries imported successfully!")

## 2. Configure AWS Bedrock Client

In [ ]:
# Configuration - Update these as needed
REGION = "us-west-2"  # Change to your region
MODEL_ID = "meta.llama3-70b-instruct-v1:0"  # Default model from textGen function

# Alternative models you can test:
# MODEL_ID = "anthropic.claude-3-sonnet-20240229-v1:0"
# MODEL_ID = "anthropic.claude-3-haiku-20240307-v1:0"

# Initialize Bedrock runtime client
bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION)

# Initialize LangChain ChatBedrock
llm = ChatBedrock(
    model_id=MODEL_ID,
    client=bedrock_runtime,
    model_kwargs={"temperature": 0, "max_tokens": 4096},
)

print(f"✅ Bedrock client initialized")
print(f"   Region: {REGION}")
print(f"   Model: {MODEL_ID}")

## 3. Define System Prompts for Each Subroute

These are the 6 system prompts you can edit and test. Modify them here, then run the test cells to see how the model responds.

**Format Notes:**
- Use Markdown formatting instructions if you want structured output
- Be specific about the assistant's role and tone
- Include any constraints (e.g., Canadian law focus, no hallucination)

In [ ]:
# =============================================================================
# SYSTEM PROMPTS - EDIT THESE TO TEST DIFFERENT BEHAVIORS
# =============================================================================

SYSTEM_PROMPTS = {
    # -------------------------------------------------------------------------
    # 1. INTAKE & FACTS
    # -------------------------------------------------------------------------
    "intake_facts": """You are a legal intake assistant helping a law student gather comprehensive case information from their client.

Your role is to:
- Help the student formulate effective questions to ask their client
- Identify missing facts that are critical to the case
- Organize gathered information into a clear factual timeline
- Flag potential inconsistencies or gaps in the client's story

Guidelines:
- Be systematic and thorough in fact-gathering
- Suggest follow-up questions based on the client's responses
- Focus on legally relevant facts
- Use plain language the student can relay to their client
- Format your responses using Markdown (use bullet lists, headers, and bold text for emphasis)

Do NOT:
- Provide legal advice or conclusions at this stage
- Make assumptions about facts not provided
- Skip over important details

Always respond in well-formatted Markdown.""",

    # -------------------------------------------------------------------------
    # 2. ISSUE IDENTIFICATION
    # -------------------------------------------------------------------------
    "issue_identification": """You are a legal issue identification assistant helping a law student identify and analyze the legal issues in their case.

Your role is to:
- Identify all potential legal issues from the facts provided
- Categorize issues by area of law (e.g., contract, tort, constitutional, criminal)
- Prioritize issues by their likelihood of success and relevance
- Explain the elements required to establish each issue
- Identify which facts support or undermine each issue

Guidelines:
- Focus on Canadian law unless otherwise specified
- Consider both federal and provincial/territorial jurisdiction
- Identify primary and secondary issues
- Note any threshold issues (standing, limitation periods, jurisdiction)
- Format your analysis using Markdown with clear headers and bullet points

Structure your response as:
## Primary Issues
## Secondary Issues
## Threshold Issues (if any)
## Recommended Focus Areas

Always cite relevant areas of law and explain your reasoning.""",

    # -------------------------------------------------------------------------
    # 3. RESEARCH STRATEGY
    # -------------------------------------------------------------------------
    "research_strategy": """You are a legal research strategy assistant helping a law student plan their legal research.

Your role is to:
- Develop a structured research plan based on identified issues
- Suggest relevant statutes, regulations, and legislative materials
- Recommend key cases and legal principles to research
- Identify secondary sources (textbooks, articles, commentary)
- Provide search terms and database recommendations

Guidelines:
- Prioritize research tasks by importance and dependency
- Focus on Canadian legal sources (CanLII, Westlaw Canada, LexisNexis Quicklaw)
- Distinguish between binding and persuasive authority
- Consider both common law and civil law sources where applicable
- Format your response using Markdown with numbered steps and resource lists

Structure your response as:
## Research Priority Order
## Primary Sources to Consult
### Legislation
### Case Law
## Secondary Sources
## Recommended Search Terms
## Research Tips

Be specific about databases and search strategies.""",

    # -------------------------------------------------------------------------
    # 4. ARGUMENT CONSTRUCTION
    # -------------------------------------------------------------------------
    "argument_construction": """You are a legal argument construction assistant helping a law student build persuasive legal arguments.

Your role is to:
- Help structure arguments using IRAC/CREAC methodology
- Identify the strongest arguments for the client's position
- Suggest how to frame facts favorably
- Recommend supporting authorities and how to use them
- Identify potential weaknesses and how to address them

Guidelines:
- Focus on clear, logical argument structure
- Use persuasive but professional legal writing style
- Connect facts directly to legal principles
- Anticipate counter-arguments
- Format arguments using Markdown with clear structure

Structure your response as:
## Main Argument
### Issue
### Rule
### Application
### Conclusion

## Alternative Arguments
## Key Authorities to Cite
## Potential Weaknesses to Address

Always explain the reasoning behind your suggestions.""",

    # -------------------------------------------------------------------------
    # 5. CONTRARIAN ANALYSIS
    # -------------------------------------------------------------------------
    "contrarian_analysis": """You are a contrarian legal analyst helping a law student stress-test their case by identifying weaknesses and opposing arguments.

Your role is to:
- Identify weaknesses in the student's position
- Construct the strongest possible opposing arguments
- Anticipate what the other side will argue
- Find holes in the factual record
- Suggest how to rebut or mitigate weaknesses

Guidelines:
- Be thorough and honest about weaknesses
- Think like opposing counsel
- Consider procedural and substantive vulnerabilities
- Identify unfavorable precedents
- Format your analysis using Markdown

Structure your response as:
## Weaknesses in Our Position
## Likely Opposing Arguments
## Unfavorable Precedents/Authority
## Factual Vulnerabilities
## Recommended Mitigation Strategies

Be candid - the goal is to strengthen the case by identifying problems early.""",

    # -------------------------------------------------------------------------
    # 6. POLICY CONTEXT
    # -------------------------------------------------------------------------
    "policy_context": """You are a legal policy analyst helping a law student understand the broader policy context of their case.

Your role is to:
- Explain the policy rationales behind relevant laws
- Identify how courts have balanced competing interests
- Discuss any recent legislative or judicial trends
- Consider Charter implications and values
- Explore how policy arguments might support the case

Guidelines:
- Connect specific rules to underlying policy goals
- Consider multiple stakeholder perspectives
- Identify evolving areas of law
- Discuss any law reform proposals or debates
- Format your analysis using Markdown

Structure your response as:
## Policy Rationale Behind the Law
## Competing Interests/Values
## Recent Trends and Developments
## Charter Considerations (if applicable)
## Policy Arguments for Our Position

Help the student see the bigger picture beyond the specific legal rules."""
}

print("✅ System prompts defined for 6 subroutes:")
for route in SYSTEM_PROMPTS.keys():
    print(f"   - {route}")

## 4. Define Chat History Structures for Each Subroute

Each subroute maintains its own conversation history. This allows the assistant to maintain context within each stage of case analysis.

In [ ]:
# =============================================================================
# CHAT HISTORIES - Separate history for each subroute
# =============================================================================

@dataclass
class ChatSession:
    """Manages chat history for a single subroute"""
    subroute: str
    history: InMemoryChatMessageHistory = field(default_factory=InMemoryChatMessageHistory)
    created_at: datetime = field(default_factory=datetime.now)
    
    def add_user_message(self, content: str):
        self.history.add_user_message(content)
    
    def add_ai_message(self, content: str):
        self.history.add_ai_message(content)
    
    def get_messages(self) -> List:
        return self.history.messages
    
    def clear(self):
        self.history.clear()
        print(f"✅ Cleared history for {self.subroute}")
    
    def display_history(self):
        print(f"\n{'='*60}")
        print(f"Chat History: {self.subroute}")
        print(f"{'='*60}")
        for i, msg in enumerate(self.history.messages):
            role = "🧑 USER" if isinstance(msg, HumanMessage) else "🤖 AI"
            content_preview = msg.content[:200] + "..." if len(msg.content) > 200 else msg.content
            print(f"\n[{i+1}] {role}:\n{content_preview}")
        if not self.history.messages:
            print("(empty)")

# Initialize 6 separate chat sessions
chat_sessions: Dict[str, ChatSession] = {
    "intake_facts": ChatSession(subroute="intake_facts"),
    "issue_identification": ChatSession(subroute="issue_identification"),
    "research_strategy": ChatSession(subroute="research_strategy"),
    "argument_construction": ChatSession(subroute="argument_construction"),
    "contrarian_analysis": ChatSession(subroute="contrarian_analysis"),
    "policy_context": ChatSession(subroute="policy_context"),
}

print("✅ Chat sessions initialized for 6 subroutes:")
for route in chat_sessions.keys():
    print(f"   - {route}")

## 5. Create Message Formatting Helper Functions

In [ ]:
# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def create_prompt_template(system_prompt: str) -> ChatPromptTemplate:
    """Create a ChatPromptTemplate with system prompt and chat history placeholder"""
    return ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}")
    ])

def format_case_context(
    case_type: str = "",
    jurisdiction: str = "",
    case_description: str = "",
    province: str = "",
    statute: str = ""
) -> str:
    """Format case context to inject into the conversation"""
    context_parts = []
    if case_type:
        context_parts.append(f"**Case Type:** {case_type}")
    if jurisdiction:
        context_parts.append(f"**Jurisdiction:** {jurisdiction}")
    if province:
        context_parts.append(f"**Province:** {province}")
    if statute:
        context_parts.append(f"**Relevant Statute:** {statute}")
    if case_description:
        context_parts.append(f"**Case Description:**\n{case_description}")
    
    return "\n".join(context_parts) if context_parts else ""

def display_response(response: str, subroute: str):
    """Pretty print the AI response"""
    print(f"\n{'='*60}")
    print(f"🤖 Response ({subroute})")
    print(f"{'='*60}")
    print(response)
    print(f"{'='*60}\n")

print("✅ Helper functions defined")

## 6. Implement Text Generation Function

This is the core function that replicates the textGen Lambda behavior (without vector store and guardrails).

In [ ]:
# =============================================================================
# CORE TEXT GENERATION FUNCTION
# =============================================================================

def generate_response(
    subroute: str,
    user_input: str,
    case_context: Optional[str] = None,
    include_history: bool = True,
    custom_system_prompt: Optional[str] = None
) -> str:
    """
    Generate a response for the specified subroute.
    
    Args:
        subroute: One of the 6 subroute keys (e.g., "intake_facts")
        user_input: The user's message/query
        case_context: Optional case context to prepend to the first message
        include_history: Whether to include chat history (default: True)
        custom_system_prompt: Override the default system prompt (for testing)
    
    Returns:
        The AI-generated response string
    """
    # Validate subroute
    if subroute not in SYSTEM_PROMPTS:
        raise ValueError(f"Invalid subroute: {subroute}. Must be one of {list(SYSTEM_PROMPTS.keys())}")
    
    # Get system prompt (use custom if provided)
    system_prompt = custom_system_prompt or SYSTEM_PROMPTS[subroute]
    
    # Get chat session
    session = chat_sessions[subroute]
    
    # Build the user message (optionally with case context)
    if case_context and len(session.get_messages()) == 0:
        # First message - include case context
        full_input = f"{case_context}\n\n---\n\n{user_input}"
    else:
        full_input = user_input
    
    # Create prompt template
    prompt = create_prompt_template(system_prompt)
    
    # Build the chain
    chain = prompt | llm
    
    # Get chat history
    history = session.get_messages() if include_history else []
    
    # Invoke the chain
    response = chain.invoke({
        "chat_history": history,
        "input": full_input
    })
    
    # Extract response content
    response_text = response.content if hasattr(response, 'content') else str(response)
    
    # Update chat history
    session.add_user_message(full_input)
    session.add_ai_message(response_text)
    
    return response_text


def chat(subroute: str, message: str, case_context: Optional[str] = None) -> None:
    """Convenience function to chat and display the response"""
    response = generate_response(subroute, message, case_context)
    display_response(response, subroute)


def reset_session(subroute: str) -> None:
    """Clear the chat history for a specific subroute"""
    chat_sessions[subroute].clear()


def reset_all_sessions() -> None:
    """Clear all chat histories"""
    for session in chat_sessions.values():
        session.clear()
    print("✅ All chat sessions cleared")


print("✅ Text generation functions defined")
print("\nUsage:")
print("  chat('intake_facts', 'Your message here')")
print("  generate_response('issue_identification', 'Your query')")
print("  reset_session('research_strategy')")
print("  reset_all_sessions()")

## 7. Sample Case Context for Testing

Define a sample case to use across all subroutes for consistent testing.

In [ ]:
# =============================================================================
# SAMPLE CASE FOR TESTING - Edit this to test with your own case
# =============================================================================

SAMPLE_CASE = format_case_context(
    case_type="Employment Law - Wrongful Dismissal",
    jurisdiction="Federal",
    province="British Columbia",
    statute="Canada Labour Code",
    case_description="""
Our client, Sarah Chen, worked as a software developer at TechCorp Inc. for 5 years.
On March 15, 2024, she was terminated without cause and offered 2 weeks of severance pay.

Key facts:
- Sarah was 35 years old at termination
- Annual salary was $120,000
- She had received positive performance reviews throughout her employment
- No written employment contract was signed
- She was given no prior warning of termination
- TechCorp claims the termination was due to "restructuring"
- Sarah noticed her position was posted on job boards 2 weeks after her termination
- She has been unable to find comparable employment after 3 months of searching

Sarah wants to know if she has a claim for wrongful dismissal and what compensation she might be entitled to.
"""
)

print("📋 Sample Case Context:")
print(SAMPLE_CASE)

---
## 8. Test: Intake & Facts

Test the intake and fact-gathering subroute.

In [ ]:
# Reset session and start fresh
reset_session("intake_facts")

# First message with case context
chat("intake_facts", 
     "I just met with a new client about a potential wrongful dismissal case. Can you help me identify what additional facts I should gather?",
     case_context=SAMPLE_CASE)

In [ ]:
# Follow-up question (uses chat history)
chat("intake_facts", 
     "The client mentioned she had some health issues in the months before termination. What questions should I ask about that?")

---
## 9. Test: Issue Identification

Test the legal issue identification subroute.

In [ ]:
# Reset session and start fresh
reset_session("issue_identification")

# Initial issue identification request
chat("issue_identification",
     "Based on the case facts provided, what are the potential legal issues I should analyze?",
     case_context=SAMPLE_CASE)

In [ ]:
# Follow-up on a specific issue
chat("issue_identification",
     "Can you elaborate on the bad faith damages issue? What elements would we need to prove?")

---
## 10. Test: Research Strategy

Test the legal research planning subroute.

In [ ]:
# Reset session and start fresh
reset_session("research_strategy")

# Initial research strategy request
chat("research_strategy",
     "I need to research the reasonable notice period for wrongful dismissal. Can you help me develop a research plan?",
     case_context=SAMPLE_CASE)

In [ ]:
# Follow-up on research strategy
chat("research_strategy",
     "What specific search terms should I use in CanLII to find cases with similar facts?")

---
## 11. Test: Argument Construction

Test the legal argument building subroute.

In [ ]:
# Reset session and start fresh
reset_session("argument_construction")

# Initial argument construction request
chat("argument_construction",
     "Help me construct an argument for why my client deserves a longer notice period than the 2 weeks offered.",
     case_context=SAMPLE_CASE)

In [ ]:
# Follow-up on argument construction
chat("argument_construction",
     "How can I strengthen the bad faith argument given the fact that her position was reposted?")

---
## 12. Test: Contrarian Analysis

Test the opposing argument and weakness identification subroute.

In [ ]:
# Reset session and start fresh
reset_session("contrarian_analysis")

# Initial contrarian analysis request
chat("contrarian_analysis",
     "Play devil's advocate. What are the strongest arguments TechCorp could make against my client's wrongful dismissal claim?",
     case_context=SAMPLE_CASE)

In [ ]:
# Follow-up on contrarian analysis
chat("contrarian_analysis",
     "What about the mitigation defense? How might TechCorp argue my client failed to mitigate her damages?")

---
## 13. Test: Policy Context

Test the policy and broader context analysis subroute.

In [ ]:
# Reset session and start fresh
reset_session("policy_context")

# Initial policy context request
chat("policy_context",
     "What are the policy rationales behind wrongful dismissal law in Canada? How might these support our case?",
     case_context=SAMPLE_CASE)

In [ ]:
# Follow-up on policy context
chat("policy_context",
     "Are there any recent trends in how courts are treating bad faith dismissal claims in the tech industry?")

---
## 14. Interactive Prompt Refinement Interface

Use this section to quickly test prompt modifications without editing the main SYSTEM_PROMPTS dictionary.

In [ ]:
# =============================================================================
# QUICK PROMPT TESTING FUNCTION
# =============================================================================

def test_prompt_variation(
    subroute: str,
    custom_prompt: str,
    test_message: str,
    case_context: Optional[str] = None
) -> str:
    """
    Test a custom prompt variation without modifying the main SYSTEM_PROMPTS.
    This creates a fresh conversation with no history.
    
    Args:
        subroute: The subroute name (for display purposes)
        custom_prompt: Your custom system prompt to test
        test_message: The user message to test with
        case_context: Optional case context
    
    Returns:
        The AI response
    """
    # Build input
    full_input = f"{case_context}\n\n---\n\n{test_message}" if case_context else test_message
    
    # Create prompt template with custom prompt
    prompt = create_prompt_template(custom_prompt)
    chain = prompt | llm
    
    # Invoke without history
    response = chain.invoke({
        "chat_history": [],
        "input": full_input
    })
    
    response_text = response.content if hasattr(response, 'content') else str(response)
    
    print(f"\n{'='*60}")
    print(f"🧪 PROMPT TEST: {subroute}")
    print(f"{'='*60}")
    print(f"\n📝 Custom Prompt Preview (first 200 chars):")
    print(f"{custom_prompt[:200]}...")
    print(f"\n💬 Test Message:")
    print(f"{test_message}")
    print(f"\n{'='*60}")
    print(f"🤖 Response:")
    print(f"{'='*60}")
    print(response_text)
    print(f"{'='*60}\n")
    
    return response_text

print("✅ Quick prompt testing function ready")
print("\nUsage:")
print("""
test_prompt_variation(
    subroute="intake_facts",
    custom_prompt="Your custom system prompt here...",
    test_message="Your test question here",
    case_context=SAMPLE_CASE  # Optional
)
""")

In [ ]:
# =============================================================================
# EXAMPLE: Testing a modified prompt
# =============================================================================

# Try a more concise issue identification prompt
test_prompt_shorter = """You are a legal issue spotter for Canadian law students.

Given case facts, identify:
1. Primary legal issues (with elements)
2. Secondary issues
3. Threshold issues (limitation periods, jurisdiction, standing)

Be concise. Use bullet points. Focus on Canadian law.
Always format your response using Markdown."""

# Uncomment to test:
# test_prompt_variation(
#     subroute="issue_identification",
#     custom_prompt=test_prompt_shorter,
#     test_message="What are the legal issues in this wrongful dismissal case?",
#     case_context=SAMPLE_CASE
# )

---
## 15. Utilities: View & Export Prompts

In [ ]:
# =============================================================================
# UTILITY FUNCTIONS
# =============================================================================

def view_prompt(subroute: str):
    """Display the current system prompt for a subroute"""
    if subroute not in SYSTEM_PROMPTS:
        print(f"❌ Invalid subroute: {subroute}")
        return
    
    print(f"\n{'='*60}")
    print(f"System Prompt: {subroute}")
    print(f"{'='*60}")
    print(SYSTEM_PROMPTS[subroute])
    print(f"{'='*60}\n")

def view_all_prompts():
    """Display all system prompts"""
    for subroute in SYSTEM_PROMPTS.keys():
        view_prompt(subroute)

def export_prompts_to_json(filepath: str = "system_prompts.json"):
    """Export all prompts to a JSON file for backup or transfer"""
    with open(filepath, 'w') as f:
        json.dump(SYSTEM_PROMPTS, f, indent=2)
    print(f"✅ Prompts exported to {filepath}")

def export_prompts_to_python(filepath: str = "system_prompts.py"):
    """Export prompts as a Python file ready to copy into Lambda"""
    with open(filepath, 'w') as f:
        f.write("# Auto-generated system prompts for text_generation Lambda\n")
        f.write("# Copy these into your Lambda function\n\n")
        f.write("SYSTEM_PROMPTS = {\n")
        for key, value in SYSTEM_PROMPTS.items():
            # Escape the prompt for Python string
            escaped = value.replace('\\', '\\\\').replace('"""', '\\"\\"\\"')
            f.write(f'    "{key}": """{escaped}""",\n\n')
        f.write("}\n")
    print(f"✅ Prompts exported to {filepath}")

def view_chat_history(subroute: str):
    """Display the chat history for a specific subroute"""
    if subroute not in chat_sessions:
        print(f"❌ Invalid subroute: {subroute}")
        return
    chat_sessions[subroute].display_history()

def view_all_histories():
    """Display chat histories for all subroutes"""
    for subroute in chat_sessions.keys():
        view_chat_history(subroute)

print("✅ Utility functions ready:")
print("  view_prompt('subroute_name')")
print("  view_all_prompts()")
print("  export_prompts_to_json()")
print("  export_prompts_to_python()")
print("  view_chat_history('subroute_name')")
print("  view_all_histories()")

In [ ]:
# View a specific prompt
view_prompt("intake_facts")

In [ ]:
# Export finalized prompts when ready
# export_prompts_to_json("finalized_prompts.json")
# export_prompts_to_python("finalized_prompts.py")

---
## 16. Quick Reference

### Subroute Keys
| Key | Description |
|-----|-------------|
| `intake_facts` | Gathering case information from client |
| `issue_identification` | Identifying legal issues |
| `research_strategy` | Planning legal research |
| `argument_construction` | Building legal arguments |
| `contrarian_analysis` | Analyzing opposing arguments |
| `policy_context` | Understanding policy implications |

### Main Functions
```python
# Chat with history
chat("subroute_key", "Your message", case_context=SAMPLE_CASE)

# Generate response (returns string)
response = generate_response("subroute_key", "Your message")

# Test custom prompt
test_prompt_variation("subroute_key", "Custom prompt", "Test message")

# Reset sessions
reset_session("subroute_key")
reset_all_sessions()

# View/Export
view_prompt("subroute_key")
export_prompts_to_python("filename.py")
```

### Tips for Prompt Refinement
1. **Be specific** about the role and tone
2. **Include format instructions** (Markdown, bullet points, headers)
3. **Add constraints** (Canadian law focus, no hallucination)
4. **Structure output** with expected sections
5. **Test edge cases** (incomplete facts, ambiguous queries)
6. **Compare responses** across different prompt versions